# Spec

> OpenAPI/Google Discovery spec parsing and provider operation loaders.

In [ ]:
#| default_exp spec

## Imports

In [ ]:
#| export
from fastcore.utils import *
from dataclasses import dataclass, field

In [ ]:
#| hide
import json, yaml
from fastcore.test import *

Converts provider specs (OpenAPI or discovery-like docs) into unified `OpSpec` records.


[The OpenAPI Specification Explained](https://learn.openapis.org/specification/)

In [ ]:
OPENAI_SPEC_URL = "https://app.stainless.com/api/spec/documented/openai/openapi.documented.yml"
GEMINI_DISCOVERY_URL = "https://generativelanguage.googleapis.com/$discovery/rest?version=v1beta"
ANTHROPIC_STATS_URL = "https://raw.githubusercontent.com/anthropics/anthropic-sdk-python/main/.stats.yml"

In [ ]:
specs_path = Path('../specs/')
specs_path.ls()

[Path('../specs/stripe.json'), Path('../specs/gemini.json'), Path('../specs/github.json'), Path('../specs/spec_manifest.json'), Path('../specs/anthropic.yml'), Path('../specs/openai.with-code-samples.yml'), Path('../specs/openapi_docs.ipynb')]

An example path `/responses` from OpenAI api spec:

```
/responses:
post:
    operationId: createResponse
    tags:
    - Responses
    summary: >
    Creates a model response. Provide [text](/docs/guides/text) or

    [image](/docs/guides/images) inputs to generate
    [text](/docs/guides/text)

    or [JSON](/docs/guides/structured-outputs) outputs. Have the model call

    your own [custom code](/docs/guides/function-calling) or use built-in

    [tools](/docs/guides/tools) like [web
    search](/docs/guides/tools-web-search)

    or [file search](/docs/guides/tools-file-search) to use your own data

    as input for the model's response.
    requestBody:
    required: true
    content:
        application/json:
        schema:
            $ref: '#/components/schemas/CreateResponse'
    responses:
    '200':
        description: OK
        content:
        application/json:
            schema:
            $ref: '#/components/schemas/Response'
        text/event-stream:
            schema:
            $ref: '#/components/schemas/ResponseStreamEvent'
    x-oaiMeta:
```

Referenced `CreateResponse` schema:

```
CreateResponse:
      allOf:
        - $ref: '#/components/schemas/CreateModelResponseProperties'
        - $ref: '#/components/schemas/ResponseProperties'
        - type: object
          properties:
            input:
              $ref: '#/components/schemas/InputParam'
            include:
              anyOf:
                - type: array
                  description: >-
                    Specify additional output data to include in the model
                    response. Currently supported values are:

                    - `web_search_call.action.sources`: Include the sources of
                    the web search tool call.

                    - `code_interpreter_call.outputs`: Includes the outputs of
                    python code execution in code interpreter tool call items.

                    - `computer_call_output.output.image_url`: Include image
                    urls from the computer call output.

                    - `file_search_call.results`: Include the search results of
                    the file search tool call.

                    - `message.input_image.image_url`: Include image urls from
                    the input message.

                    - `message.output_text.logprobs`: Include logprobs with
                    assistant messages.

                    - `reasoning.encrypted_content`: Includes an encrypted
                    version of reasoning tokens in reasoning item outputs. This
                    enables reasoning items to be used in multi-turn
                    conversations when using the Responses API statelessly (like
                    when the `store` parameter is set to `false`, or when an
                    organization is enrolled in the zero data retention
                    program).
                  items:
                    $ref: '#/components/schemas/IncludeEnum'
                - type: 'null'
            parallel_tool_calls:
              anyOf:
                - type: boolean
                  description: |
                    Whether to allow the model to run tool calls in parallel.
                  default: true
                - type: 'null'
            store:
              anyOf:
                - type: boolean
                  description: >
                    Whether to store the generated model response for later
                    retrieval via

                    API.
                  default: true
                - type: 'null'
            instructions:
              anyOf:
                - type: string
                  description: >
                    A system (or developer) message inserted into the model's
                    context.


                    When using along with `previous_response_id`, the
                    instructions from a previous

                    response will not be carried over to the next response. This
                    makes it simple

                    to swap out system (or developer) messages in new responses.
                - type: 'null'
            stream:
              anyOf:
                - description: >
                    If set to true, the model response data will be streamed to
                    the client

                    as it is generated using [server-sent
                    events](https://developer.mozilla.org/en-US/docs/Web/API/Server-sent_events/Using_server-sent_events#Event_stream_format).

                    See the [Streaming section
                    below](/docs/api-reference/responses-streaming)

                    for more information.
                  type: boolean
                  default: false
                - type: 'null'
            stream_options:
              $ref: '#/components/schemas/ResponseStreamOptions'
            conversation:
              anyOf:
                - $ref: '#/components/schemas/ConversationParam'
                - type: 'null'
            context_management:
              anyOf:
                - type: array
                  description: |
                    Context management configuration for this request.
                  minItems: 1
                  items:
                    $ref: '#/components/schemas/ContextManagementParam'
                - type: 'null'
```                

### `OpSpec`

Canonical operation metadata record used to generate dynamic API methods

In [ ]:
from ghapi.core import GhApi
g = GhApi()

In [ ]:
g.gists

- [gists.list](https://docs.github.com/rest/gists/gists#list-gists-for-the-authenticated-user)(since, per_page, page): *List gists for the authenticated user*
- [gists.create](https://docs.github.com/rest/gists/gists#create-a-gist)(description, files, public): *Create a gist*
- [gists.list_public](https://docs.github.com/rest/gists/gists#list-public-gists)(since, per_page, page): *List public gists*
- [gists.list_starred](https://docs.github.com/rest/gists/gists#list-starred-gists)(since, per_page, page): *List starred gists*
- [gists.get](https://docs.github.com/rest/gists/gists#get-a-gist)(gist_id): *Get a gist*
- [gists.update](https://docs.github.com/rest/gists/gists#update-a-gist)(gist_id, description, files): *Update a gist*
- [gists.delete](https://docs.github.com/rest/gists/gists#delete-a-gist)(gist_id): *Delete a gist*
- [gists.list_comments](https://docs.github.com/rest/gists/comments#list-gist-comments)(gist_id, per_page, page): *List gist comments*
- [gists.create_comment](https://docs.github.com/rest/gists/comments#create-a-gist-comment)(gist_id, body): *Create a gist comment*
- [gists.get_comment](https://docs.github.com/rest/gists/comments#get-a-gist-comment)(gist_id, comment_id): *Get a gist comment*
- [gists.update_comment](https://docs.github.com/rest/gists/comments#update-a-gist-comment)(gist_id, comment_id, body): *Update a gist comment*
- [gists.delete_comment](https://docs.github.com/rest/gists/comments#delete-a-gist-comment)(gist_id, comment_id): *Delete a gist comment*
- [gists.list_commits](https://docs.github.com/rest/gists/gists#list-gist-commits)(gist_id, per_page, page): *List gist commits*
- [gists.list_forks](https://docs.github.com/rest/gists/gists#list-gist-forks)(gist_id, per_page, page): *List gist forks*
- [gists.fork](https://docs.github.com/rest/gists/gists#fork-a-gist)(gist_id): *Fork a gist*
- [gists.check_is_starred](https://docs.github.com/rest/gists/gists#check-if-a-gist-is-starred)(gist_id): *Check if a gist is starred*
- [gists.star](https://docs.github.com/rest/gists/gists#star-a-gist)(gist_id): *Star a gist*
- [gists.unstar](https://docs.github.com/rest/gists/gists#unstar-a-gist)(gist_id): *Unstar a gist*
- [gists.get_revision](https://docs.github.com/rest/gists/gists#get-a-gist-revision)(gist_id, sha): *Get a gist revision*
- [gists.list_for_user](https://docs.github.com/rest/gists/gists#list-gists-for-a-user)(username, since, per_page, page): *List gists for a user*

In [ ]:
#| export
@dataclass(frozen=True)
class OpSpec:
    "Operation metadata used by the dynamic client layer."
    group: str
    name: str
    path: str
    verb: str
    summary: str = ""
    route_params: List[str] = field(default_factory=list) # https://learn.openapis.org/specification/parameters#the-parameter-object
    query_params: List[str] = field(default_factory=list) # https://learn.openapis.org/specification/parameters#the-parameter-object
    body_params: List[str] = field(default_factory=list)
    file_params: List[str] = field(default_factory=list)  # format: binary params (multipart file uploads)
    required_params: List[str] = field(default_factory=list)
    param_types: Dict = field(default_factory=dict)
    param_defaults: Dict = field(default_factory=dict)
    param_docs: Dict = field(default_factory=dict)
    docs_url: str = ""
    
    def mk_doc(self):
        rows = []
        for f,v in vars(self).items():
            if f not in ('param_types','param_defaults','param_docs'): rows.append(f'| `{f}` | {v} |')
        md = f'| Field | Value |\n|---|---|\n' + '\n'.join(rows)
        all_params = self.route_params + self.query_params + self.body_params
        if all_params:
            md += f'\n\n| Param | Type | Default | Required | Description |\n|---|---|---|---|---|\n'
            for p in all_params:
                t = self.param_types.get(p, '')
                t = t.__name__ if isinstance(t, type) else str(t)
                d = self.param_defaults.get(p, '')
                r = '✓' if p in self.required_params else ''
                doc = self.param_docs.get(p, '')
                md += f'| `{p}` | {t} | {d} | {r} | {doc} |\n'
        return md

    def __post_init__(self): object.__setattr__(self, '__doc__', self.mk_doc())
    def _repr_markdown_(self): return self.__doc__

TODO: maybe add to fastcore

In [ ]:
@patch
def read_yaml(self:Path, encoding=None, errors=None):
    "Same as `read_text` followed by `yaml.safe_load`"
    return yaml.safe_load(self.read_text(encoding=encoding, errors=errors))

In [ ]:
oai_spec = dict2obj((specs_path/'openai.with-code-samples.yml').read_yaml())

In [ ]:
oai_spec.keys()

dict_keys(['openapi', 'info', 'servers', 'security', 'tags', 'paths', 'webhooks', 'components', 'x-oaiMeta'])

In [ ]:
oai_spec.servers, len(oai_spec.paths)

([{'url': 'https://api.openai.com/v1'}], 161)

In [ ]:
ant_spec = dict2obj((specs_path/'anthropic.yml').read_yaml())

In [ ]:
ant_spec.keys()

dict_keys(['openapi', 'info', 'paths', 'components', 'servers'])

In [ ]:
ant_spec.servers, len(ant_spec.paths)

([{'url': 'https://api.anthropic.com'}], 31)

In [ ]:
gh_spec = dict2obj((specs_path/'github.json').read_json())
stripe_spec = dict2obj((specs_path/'stripe.json').read_json())
gh_spec.keys(), stripe_spec.keys()

(dict_keys(['openapi', 'info', 'tags', 'servers', 'externalDocs', 'paths', 'x-webhooks', 'components']),
 dict_keys(['components', 'info', 'openapi', 'paths', 'security', 'servers']))

### Module State

In [ ]:
#| export
_http_verbs = {"get", "post", "put", "patch", "delete", "options", "head"}
_pat_non_alnum = re.compile(r"[^a-zA-Z0-9]+")
_pat_path_param = re.compile(r"\{([^}]+)\}")

## OpenAPI

### `snake`

TODO: maybe add to fastcore

In [ ]:
#| export
def snake(s: str):
    "Convert an identifier-ish string to snake_case."
    s = _pat_non_alnum.sub("_", s).strip("_")
    s = re.sub(r"(.)([A-Z][a-z]+)", r"\1_\2", s)
    s = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", s)
    return s.lower().strip("_")

In [ ]:
op_id = oai_spec.paths['/responses']['post']['operationId']
op_id, snake(op_id)

('createResponse', 'create_response')

### `_group_name`

Infer group + name from operationId with path fallback.

In [ ]:
#| export
def _group_name(op_id: str, path: str, http_verb:str='', op_id_seps=('/', '.')):
    "Infer group + name from operationId with path fallback."
    for sep in op_id_seps:
        if sep in op_id:
            grp, nm = op_id.split(sep, 1)
            return snake(grp), snake(nm)
    segs = [s for s in path.strip("/").split("/") if s and not s.startswith("{")]
    # strip version /v1,/v1.2,etc
    if segs and re.fullmatch(r"v\d+(?:[a-zA-Z0-9._-]*)?", segs[0]): segs = segs[1:] 
    grp = snake(segs[0]) if segs else "api"
    return grp, snake(op_id or ("_".join([http_verb.lower()]+segs[1:]) if segs else "call"))

Here is the doc: https://learn.openapis.org/specification/tags.html#basic-tag-usage-all-versions

In [ ]:
L([nested_idx(v,'post', 'operationId') for k,v in oai_spec.paths.items()]).filter()[:5]

['createAssistant', 'modifyAssistant', 'createSpeech', 'createTranscription', 'createTranslation']

In [ ]:
# ghapi-style (slash-separated)
test_eq(_group_name('meta/root', '/'), ('meta', 'root'))
test_eq(_group_name('apps/get-authenticated', '/app'), ('apps', 'get_authenticated'))
test_eq(_group_name('apps/list-webhook-deliveries', '/app/hook/deliveries'), ('apps', 'list_webhook_deliveries'))
test_eq(_group_name('security-advisories/list-global-advisories', '/advisories'), ('security_advisories', 'list_global_advisories'))

# Stripe-style (PascalCase, no separator — falls back to path)
test_eq(_group_name('GetAccount', '/v1/account'), ('account', 'get_account'))
test_eq(_group_name('PostAccountLinks', '/v1/account_links'), ('account_links', 'post_account_links'))
test_eq(_group_name('DeleteAccountsAccount', '/v1/accounts/{account}'), ('accounts', 'delete_accounts_account'))

# OpenAI-style (camelCase, no separator — falls back to path)
test_eq(_group_name('createResponse', '/responses'), ('responses', 'create_response'))
test_eq(_group_name('createChatCompletion', '/chat/completions'), ('chat', 'create_chat_completion'))
test_eq(_group_name('createSpeech', '/audio/speech'), ('audio', 'create_speech'))
test_eq(_group_name('listModels', '/models'), ('models', 'list_models'))

# Anthropic-style (snake_case, no separator — falls back to path)
test_eq(_group_name('messages_post', '/v1/messages'), ('messages', 'messages_post'))
test_eq(_group_name('message_batches_post', '/v1/messages/batches'), ('messages', 'message_batches_post'))
test_eq(_group_name('upload_file_v1_files_post', '/v1/files'), ('files', 'upload_file_v1_files_post'))

# Google Discovery-style (dot-separated)
test_eq(_group_name('models.list', '/models'), ('models', 'list'))
test_eq(_group_name('models.get', '/models/{model}'), ('models', 'get'))

# Edge: no operationId and nested path prepends http verb too
test_eq(_group_name('', '/v1/messages', 'post'), ('messages', 'post'))
test_eq(_group_name('', '/users/{id}/posts', 'GET'), ('users', 'get_posts'))
test_eq(_group_name('', '/containers/{container_id}/files/{file_id}/content', 'get'),
        ('containers', 'get_files_content'))

In [ ]:
op_id_paths = L(gh_spec.paths.items()).map(lambda o: (nested_idx(o[1], 'post', 'operationId'),o[0])).filter(lambda o:o[0])
list(op_id_paths[:5])

[('agent-tasks/create-task', '/agents/repos/{owner}/{repo}/tasks'),
 ('apps/create-from-manifest', '/app-manifests/{code}/conversions'),
 ('apps/redeliver-webhook-delivery',
  '/app/hook/deliveries/{delivery_id}/attempts'),
 ('apps/create-installation-access-token',
  '/app/installations/{installation_id}/access_tokens'),
 ('apps/check-token', '/applications/{client_id}/token')]

In [ ]:
list(op_id_paths.map(lambda o: _group_name(*o))[:5])

[('agent_tasks', 'create_task'),
 ('apps', 'create_from_manifest'),
 ('apps', 'redeliver_webhook_delivery'),
 ('apps', 'create_installation_access_token'),
 ('apps', 'check_token')]

In [ ]:
op_id_paths = L(stripe_spec.paths.items()).map(lambda o: (nested_idx(o[1], 'post', 'operationId'),o[0])).filter(lambda o:o[0])
list(op_id_paths[:5])

[('PostAccountLinks', '/v1/account_links'),
 ('PostAccountSessions', '/v1/account_sessions'),
 ('PostAccounts', '/v1/accounts'),
 ('PostAccountsAccount', '/v1/accounts/{account}'),
 ('PostAccountsAccountBankAccounts', '/v1/accounts/{account}/bank_accounts')]

In [ ]:
list(op_id_paths.map(lambda o: _group_name(*o))[:5])

[('account_links', 'post_account_links'),
 ('account_sessions', 'post_account_sessions'),
 ('accounts', 'post_accounts'),
 ('accounts', 'post_accounts_account'),
 ('accounts', 'post_accounts_account_bank_accounts')]

In [ ]:
op_id_paths = L(oai_spec.paths.items()).map(lambda o: (nested_idx(o[1], 'post', 'operationId'),o[0])).filter(lambda o:o[0])
list(op_id_paths[:5])

[('createAssistant', '/assistants'),
 ('modifyAssistant', '/assistants/{assistant_id}'),
 ('createSpeech', '/audio/speech'),
 ('createTranscription', '/audio/transcriptions'),
 ('createTranslation', '/audio/translations')]

In [ ]:
list(op_id_paths.map(lambda o: _group_name(*o))[:5])

[('assistants', 'create_assistant'),
 ('assistants', 'modify_assistant'),
 ('audio', 'create_speech'),
 ('audio', 'create_transcription'),
 ('audio', 'create_translation')]

### `_path_params`

Extract route params from /x/{id} paths.

https://learn.openapis.org/specification/parameters.html

In [ ]:
#| export
def _path_params(path: str):
    "Extract route params from /x/{id} paths."
    return [snake(o.lstrip("+")) for o in _pat_path_param.findall(path)]

In [ ]:
pparams = L(oai_spec.paths.keys()).map(lambda o: (o, _path_params(o))).filter(lambda o: o[1])
for o in pparams[:4]: print(f"{o[0]} --> {o[1]}")

/assistants/{assistant_id} --> ['assistant_id']
/audio/voice_consents/{consent_id} --> ['consent_id']
/batches/{batch_id} --> ['batch_id']
/batches/{batch_id}/cancel --> ['batch_id']


In [ ]:
pparams = L(ant_spec.paths.keys()).map(lambda o: (o, _path_params(o))).filter(lambda o: o[1])
for o in pparams[:4]: print(f"{o[0]} --> {o[1]}")

/v1/models/{model_id} --> ['model_id']
/v1/messages/batches/{message_batch_id} --> ['message_batch_id']
/v1/messages/batches/{message_batch_id}/cancel --> ['message_batch_id']
/v1/messages/batches/{message_batch_id}/results --> ['message_batch_id']


### `_resolve_ref`

Resolve a local #/components/.

In [ ]:
#| export
def _resolve_ref(ref, spec):
    'Resolve a reference object from OpenAPI or Discovery spec'
    if ref.startswith('#/'): return nested_idx(spec, *ref.lstrip('#/').split('/'))
    else: return nested_idx(spec, 'schemas', ref)

### `_resolve_obj`

Resolve a `$ref` object and merge local overrides when present.

- https://spec.openapis.org/oas/latest#operation-object
- https://spec.openapis.org/oas/latest#parameter-object
- https://spec.openapis.org/oas/latest#request-body-object

In [ ]:
#| export
def _resolve_obj(obj, spec):
    "Resolve a `$ref` object and merge local overrides when present."
    if "$ref" not in obj: return obj
    base = _resolve_ref(obj["$ref"], spec)
    if len(obj) == 1: return base
    merged = dict(base)
    merged.update({k: v for k, v in obj.items() if k != "$ref"})
    return merged

In [ ]:
obj = {'type': 'string', 'description': 'A user name'}
_resolve_obj(obj,oai_spec)

{'type': 'string', 'description': 'A user name'}

In [ ]:
obj = {'$ref': '#/components/schemas/CreateResponse'}
list(_resolve_obj(obj,oai_spec)['allOf'])[:2]

[{'$ref': '#/components/schemas/CreateModelResponseProperties'},
 {'$ref': '#/components/schemas/ResponseProperties'}]

In [ ]:
obj = {
    '$ref': '#/components/schemas/CreateResponse',
    'description': 'Custom description for this specific usage'
}
obj = _resolve_obj(obj,oai_spec)
list(obj['allOf'])[:2], obj['description']

([{'$ref': '#/components/schemas/CreateModelResponseProperties'},
  {'$ref': '#/components/schemas/ResponseProperties'}],
 'Custom description for this specific usage')

### `_clean_desc`

Normalize a description string to a compact one-liner.

In [ ]:
#| export
def _clean_desc(v):
    "Normalize a description string to a compact one-liner."
    if not isinstance(v, str):
        return ""
    return " ".join(v.strip().split())

### `_schema_props_required`

In [ ]:
#| export
_pat_req = re.compile(r'^Required\.', re.IGNORECASE)

def _schema_props_required(schema, spec):
    "Collect properties and required fields from a schema dict."
    if not isinstance(schema, dict): return {}, set()
    schema = _resolve_obj(schema, spec)
    props = schema.get("properties", {})
    req = set(schema.get("required", []))
    if not req: 
        req = {k for k,v in props.items() if _pat_req.match(v.get("description", ""))}
    for sub in schema.get("allOf", []):
        p, r = _schema_props_required(sub, spec)
        props = merge(props, p)
        req |= r
    return props, req

In [ ]:
post_sch_path = ('post', 'requestBody', 'content', 'application/json', 'schema')
oai_post_schs = L(oai_spec.paths.items()).map(lambda o: (nested_idx(o[1], *post_sch_path),o[0]))
oai_post_schs = oai_post_schs.filter(lambda o:o[0])
list(oai_post_schs[:3])

[({'$ref': '#/components/schemas/CreateAssistantRequest'}, '/assistants'),
 ({'$ref': '#/components/schemas/ModifyAssistantRequest'},
  '/assistants/{assistant_id}'),
 ({'$ref': '#/components/schemas/CreateSpeechRequest'}, '/audio/speech')]

In [ ]:
props, req = _schema_props_required(oai_post_schs[0][0], spec=oai_spec)
props.keys(), req

(dict_keys(['model', 'name', 'description', 'instructions', 'reasoning_effort', 'tools', 'tool_resources', 'metadata', 'temperature', 'top_p', 'response_format']),
 {'model'})

In [ ]:
post_sch_path = ('post', 'requestBody', 'content', 'application/json', 'schema')
ant_post_schs = L(ant_spec.paths.items()).map(lambda o: (nested_idx(o[1], *post_sch_path),o[0]))
ant_post_schs = ant_post_schs.filter(lambda o:o[0])
list(ant_post_schs[:3])

[({'$ref': '#/components/schemas/CreateMessageParams'}, '/v1/messages'),
 ({'$ref': '#/components/schemas/CompletionRequest'}, '/v1/complete'),
 ({'$ref': '#/components/schemas/CreateMessageBatchParams'},
  '/v1/messages/batches')]

In [ ]:
props, req = _schema_props_required(ant_post_schs[0][0], spec=ant_spec)
props.keys(), req

(dict_keys(['model', 'messages', 'cache_control', 'container', 'inference_geo', 'max_tokens', 'metadata', 'output_config', 'service_tier', 'stop_sequences', 'stream', 'system', 'temperature', 'thinking', 'tool_choice', 'tools', 'top_k', 'top_p']),
 {'max_tokens', 'messages', 'model'})

In [ ]:
sample = L(stripe_spec.paths.items()).filter(lambda o: nested_idx(o[1], 'post', 'requestBody'))[:1]
nested_idx(sample[0][1], 'post', 'requestBody', 'content').keys()

dict_keys(['application/x-www-form-urlencoded'])

In [ ]:
post_sch_path = ('post', 'requestBody', 'content', 'application/x-www-form-urlencoded', 'schema')
stripe_post_schs = L(stripe_spec.paths.items()).map(lambda o: (nested_idx(o[1], *post_sch_path),o[0]))
stripe_post_schs = stripe_post_schs.filter(lambda o:o[0])
list(stripe_post_schs[:1])

[({'additionalProperties': False,
   'properties': {'account': {'description': 'The identifier of the account to create an account link for.',
     'maxLength': 5000,
     'type': 'string'},
    'collect': {'description': 'The collect parameter is deprecated. Use `collection_options` instead.',
     'enum': ['currently_due', 'eventually_due'],
     'type': 'string'},
    'collection_options': {'description': 'Specifies the requirements that Stripe collects from connected accounts in the Connect Onboarding flow.',
     'properties': {'fields': {'enum': ['currently_due', 'eventually_due'],
       'type': 'string'},
      'future_requirements': {'enum': ['include', 'omit'], 'type': 'string'}},
     'title': 'collection_options_params',
     'type': 'object'},
    'expand': {'description': 'Specifies which fields in the response should be expanded.',
     'items': {'maxLength': 5000, 'type': 'string'},
     'type': 'array'},
    'refresh_url': {'description': "The URL the user will be redi

In [ ]:
props, req = _schema_props_required(stripe_post_schs[0][0], spec=stripe_spec)
props.keys(), req

(dict_keys(['account', 'collect', 'collection_options', 'expand', 'refresh_url', 'return_url', 'type']),
 {'account', 'type'})

### `_schema_py_type`

Best-effort Python type mapping from JSON schema fragments.

In [ ]:
#| export
_type_map = dict(string=str, integer=int, number=float, boolean=bool, array=list, object=dict)

def _schema_py_type(schema, spec):
    "Best-effort Python type from a JSON Schema fragment."
    if not isinstance(schema, dict): return None
    schema = _resolve_obj(schema, spec)
    t = schema.get("type")
    if t == "string" and schema.get("format") == "binary": return bytes
    if t in _type_map: return _type_map[t]
    if t == "null": return type(None)
    for key in ("oneOf", "anyOf", "allOf"):
        types = L(schema.get(key, [])).map(partial(_schema_py_type, spec=spec)).filter().unique()
        non_none = [o for o in types if o is not type(None)]
        if len(non_none) == 1: return non_none[0]

In [ ]:
props, req = _schema_props_required(oai_post_schs[0][0], spec=oai_spec)
ptypes = {k: _schema_py_type(v, oai_spec) for k,v in props.items()}
ptypes, req

({'model': str,
  'name': str,
  'description': str,
  'instructions': str,
  'reasoning_effort': str,
  'tools': list,
  'tool_resources': dict,
  'metadata': dict,
  'temperature': float,
  'top_p': float,
  'response_format': None},
 {'model'})

In [ ]:
props, req = _schema_props_required(ant_post_schs[0][0], spec=ant_spec)
ptypes = {k: _schema_py_type(v, ant_spec) for k,v in props.items()}
ptypes, req

({'model': str,
  'messages': list,
  'cache_control': dict,
  'container': str,
  'inference_geo': str,
  'max_tokens': int,
  'metadata': dict,
  'output_config': dict,
  'service_tier': str,
  'stop_sequences': list,
  'stream': bool,
  'system': None,
  'temperature': float,
  'thinking': dict,
  'tool_choice': dict,
  'tools': list,
  'top_k': int,
  'top_p': float},
 {'max_tokens', 'messages', 'model'})

In [ ]:
props, req = _schema_props_required(stripe_post_schs[0][0], spec=stripe_spec)
ptypes = {k: _schema_py_type(v, stripe_spec) for k,v in props.items()}
ptypes, req

({'account': str,
  'collect': str,
  'collection_options': dict,
  'expand': list,
  'refresh_url': str,
  'return_url': str,
  'type': str},
 {'account', 'type'})

### `_collect_params`

Collect route and query params from operation + path level params.

In [ ]:
#| export
_MISSING = object()

def _prop_default(v, spec):
    "Get default value from a property schema, resolving refs and anyOf/oneOf."
    v = _resolve_obj(v, spec)
    d = v.get("default", _MISSING)
    if d is not _MISSING: return d
    for key in ("anyOf", "oneOf"):
        has_null = False
        for sub in v.get(key, []):
            sub = _resolve_obj(sub, spec)
            if sub.get("type") == "null": has_null = True
            elif sub.get("default", _MISSING) is not _MISSING: return sub["default"]
        if has_null: return None  # nullable → default None
    return _MISSING

In [ ]:
#| export
def _collect_params(op, path_desc, spec):
    "Collect route and query params from operation + path level params."
    route, query, req, ptypes, pdocs, defaults = [], [], set(), {}, {}, {}
    for p in op.get("parameters", []) + path_desc.get("parameters", []):
        p = _resolve_obj(p, spec)
        nm, where = str(p.get("name", "")).lstrip("+"), p.get("in")
        if not nm: continue
        if where == "path" and nm not in route:
            route.append(nm); req.add(nm)
        elif where == "query" and nm not in query:
            query.append(nm)
            if p.get("required"): req.add(nm)
        sch = p.get("schema")
        if isinstance(sch, dict):
            ptypes[nm] = _schema_py_type(sch, spec)
            d = _prop_default(sch, spec)
            if d is not _MISSING: defaults[nm] = d
        desc = _clean_desc(p.get("description"))
        if desc: pdocs[nm] = desc
    return AttrDict(route_params=route, query_params=query, required_params=req, 
                    param_types=ptypes, param_docs=pdocs, param_defaults=defaults)

In [ ]:
path_desc = oai_spec.paths['/batches/{batch_id}/cancel']

In [ ]:
_collect_params(path_desc.post, path_desc, oai_spec)

```python
{ 'param_defaults': {},
  'param_docs': {'batch_id': 'The ID of the batch to cancel.'},
  'param_types': {'batch_id': <class 'str'>},
  'query_params': [],
  'required_params': {'batch_id'},
  'route_params': ['batch_id']}
```

In [ ]:
path_desc = ant_spec.paths['/v1/files/{file_id}/content?beta=true']

In [ ]:
_collect_params(path_desc['get'], path_desc, ant_spec)

```python
{ 'param_defaults': {},
  'param_docs': { 'anthropic-beta': 'Optional header to specify the beta '
                                    'version(s) you want to use. To use '
                                    'multiple betas, use a comma separated '
                                    'list like `beta1,beta2` or specify the '
                                    'header multiple times for each beta.',
                  'anthropic-version': 'The version of the Claude API you want '
                                       'to use. Read more about versioning and '
                                       'our version history '
                                       '[here](https://docs.claude.com/en/api/versioning).',
                  'file_id': 'ID of the File.',
                  'x-api-key': 'Your unique API key for authentication. This '
                               'key is required in the header of all API '
                               'requests, to authenticate your account and '
                               "access Anthropic's services. Get your API key "
                               'through the '
                               '[Console](https://console.anthropic.com/settings/keys). '
                               'Each key is scoped to a Workspace.'},
  'param_types': { 'anthropic-beta': <class 'str'>,
                   'anthropic-version': <class 'str'>,
                   'file_id': <class 'str'>,
                   'x-api-key': <class 'str'>},
  'query_params': [],
  'required_params': {'file_id'},
  'route_params': ['file_id']}
```

### `_body_params`

Extract request JSON/body params from requestBody schema.

**TODO**: Refactor ref resolution — currently `_resolve_obj`/`_resolve_ref` are called ad-hoc in multiple places (`_collect_params`, `_body_params`, `_schema_props_required`, `_schema_py_type`, `_prop_desc`). Consider a single upfront pass that denormalizes all `$ref`s in the spec tree, so downstream functions work with plain dicts. Also audit for unresolved ref edge cases like `_prop_desc` (where description was nested inside `anyOf`/`oneOf` children).

In [ ]:
# oai_props_req = L(oai_rb_schemas).map(partial(_schema_props_required, spec=oai_spec))
# ant_props_req = L(ant_rb_schemas).map(partial(_schema_props_required, spec=ant_spec))

In [ ]:
#| export
def _prop_desc(v, spec):
    "Get description from a property schema, resolving refs and anyOf/oneOf."
    v = _resolve_obj(v, spec)
    d = v.get("description")
    if d: return _clean_desc(d)
    for key in ("anyOf", "oneOf"):
        for sub in v.get(key, []):
            sub = _resolve_obj(sub, spec)
            if sub.get("type") != "null" and sub.get("description"):
                return _clean_desc(sub["description"])
    return ""

In [ ]:
#| export
ctypes = ("application/json", "application/x-www-form-urlencoded", "multipart/form-data")

def _body_params(op, spec):
    "Extract request JSON/body params from requestBody schema."
    rb = _resolve_obj(op.get("requestBody", {}), spec)
    content = rb.get("content", {})
    schema = first((content.get(ct, {}).get("schema") for ct in ctypes), noop)
    if not schema:
        return AttrDict(body_params=[], file_params=[], required_params=set(), param_types={}, param_docs={}, param_defaults={})
    props, req = _schema_props_required(schema, spec)
    fparams = [k for k,v in props.items() if _resolve_obj(v, spec).get("format") == "binary"]
    bparams = [k for k in props if k not in fparams]
    ptypes = {k: _schema_py_type(v, spec) for k,v in props.items()}
    pdocs = {k: d for k,v in props.items() if (d := _prop_desc(v, spec))}
    defaults = {k: d for k,v in props.items() if (d := _prop_default(v, spec)) is not _MISSING}
    # Params without a default or nullable type are required
    # req |= {k for k in props if k not in defaults} # too aggresive doesn't match when spec is incomplete
    return AttrDict(body_params=bparams, file_params=fparams, required_params=req, param_types=ptypes, 
                    param_docs=pdocs, param_defaults=defaults)

In [ ]:
# _body_params(oai_spec.paths['/assistants'].post, oai_spec)

In [ ]:
# _body_params(ant_spec.paths['/v1/messages'].post, ant_spec)

### `_first_url`

Extract first URL from markdown/plain text.

In [ ]:
#| export
_pat_md_url = re.compile(r"\[[^\]]+\]\((https?://[^)\s]+)\)")
_pat_url = re.compile(r"(https?://[^\s)>\"]+)")

def _first_url(text):
    "Extract first URL from markdown/plain text."
    m = _pat_md_url.search(text)
    if m: return m.group(1).rstrip(".,;")
    m = _pat_url.search(text)
    if m: return m.group(1).rstrip(".,;")
    return ""

### `_op_docs_url`

Best-effort operation docs URL extraction.

In [ ]:
#| export
def _op_docs_url(op):
    desc = op.get("description")
    if desc and (durl := _first_url(desc)): return durl

In [ ]:
# _op_docs_url(pdesc['get'])

### `openapi_to_ops`

Converts OpenAPI-like specs into a list of `OpSpec` records

In [ ]:
#| export
def openapi_to_ops(spec):
    "Convert OpenAPI-like dict to OpSpec list."
    res = []
    for path, path_desc in spec.paths.items():
        for verb, op in path_desc.items():
            if verb.lower() not in _http_verbs: continue
            op_id = str(op.get("operationId") or f"{verb}_{path}")
            group, name = _group_name(op_id, path, verb)
            pdict, bpdict = _collect_params(op, path_desc, spec), _body_params(op, spec)
            res.append(
                OpSpec(
                    group=group,
                    name=name,
                    path=path,
                    verb=verb.upper(),
                    summary=str(op.get("summary") or ""),
                    route_params=pdict.route_params or _path_params(path),
                    query_params=pdict.query_params,
                    body_params=bpdict.body_params,
                    file_params=bpdict.file_params,
                    required_params=pdict.required_params | bpdict.required_params,
                    param_types=merge(pdict.param_types, bpdict.param_types),
                    param_defaults=merge(pdict.param_defaults, bpdict.param_defaults),
                    param_docs=merge(pdict.param_docs, bpdict.param_docs),
                    docs_url=_op_docs_url(op)
                )
            )
    return res

In [ ]:
oai_ops = openapi_to_ops(oai_spec)
ant_ops = openapi_to_ops(ant_spec)
gh_ops  = openapi_to_ops(gh_spec)
stripe_ops = openapi_to_ops(stripe_spec)

In [ ]:
print(f"OpenAI: {len(oai_ops)}, Anthropic: {len(ant_ops)}, GitHub: {len(gh_ops)} ops, Stripe: {len(stripe_ops)} ops")

OpenAI: 241, Anthropic: 47, GitHub: 1112 ops, Stripe: 587 ops


In [ ]:
gh_ops[10]

| Field | Value |
|---|---|
| `group` | apps |
| `name` | get_webhook_config_for_app |
| `path` | /app/hook/config |
| `verb` | GET |
| `summary` | Get a webhook configuration for an app |
| `route_params` | [] |
| `query_params` | [] |
| `body_params` | [] |
| `file_params` | [] |
| `required_params` | set() |
| `docs_url` | https://docs.github.com/apps/building-github-apps/authenticating-with-github-apps/#authenticating-as-a-github-app |

In [ ]:
stripe_ops[10]

| Field | Value |
|---|---|
| `group` | accounts |
| `name` | get_accounts_account_bank_accounts_id |
| `path` | /v1/accounts/{account}/bank_accounts/{id} |
| `verb` | GET |
| `summary` | Retrieve an external account |
| `route_params` | ['account', 'id'] |
| `query_params` | ['expand'] |
| `body_params` | [] |
| `file_params` | [] |
| `required_params` | {'id', 'account'} |
| `docs_url` | None |

| Param | Type | Default | Required | Description |
|---|---|---|---|---|
| `account` | str |  | ✓ |  |
| `id` | str |  | ✓ | Unique identifier for the external account to be retrieved. |
| `expand` | list |  |  | Specifies which fields in the response should be expanded. |


In [ ]:
oai_ops[0]

| Field | Value |
|---|---|
| `group` | assistants |
| `name` | list_assistants |
| `path` | /assistants |
| `verb` | GET |
| `summary` | Returns a list of assistants. |
| `route_params` | [] |
| `query_params` | ['limit', 'order', 'after', 'before'] |
| `body_params` | [] |
| `file_params` | [] |
| `required_params` | set() |
| `docs_url` | None |

| Param | Type | Default | Required | Description |
|---|---|---|---|---|
| `limit` | int | 20 |  | A limit on the number of objects to be returned. Limit can range between 1 and 100, and the default is 20. |
| `order` | str | desc |  | Sort order by the `created_at` timestamp of the objects. `asc` for ascending order and `desc` for descending order. |
| `after` | str |  |  | A cursor for use in pagination. `after` is an object ID that defines your place in the list. For instance, if you make a list request and receive 100 objects, ending with obj_foo, your subsequent call can include after=obj_foo in order to fetch the next page of the list. |
| `before` | str |  |  | A cursor for use in pagination. `before` is an object ID that defines your place in the list. For instance, if you make a list request and receive 100 objects, starting with obj_foo, your subsequent call can include before=obj_foo in order to fetch the previous page of the list. |


In [ ]:
ant_ops[0]

| Field | Value |
|---|---|
| `group` | messages |
| `name` | messages_post |
| `path` | /v1/messages |
| `verb` | POST |
| `summary` | Create a Message |
| `route_params` | [] |
| `query_params` | [] |
| `body_params` | ['model', 'messages', 'cache_control', 'container', 'inference_geo', 'max_tokens', 'metadata', 'output_config', 'service_tier', 'stop_sequences', 'stream', 'system', 'temperature', 'thinking', 'tool_choice', 'tools', 'top_k', 'top_p'] |
| `file_params` | [] |
| `required_params` | {'messages', 'max_tokens', 'model'} |
| `docs_url` | https://docs.claude.com/en/docs/initial-setup |

| Param | Type | Default | Required | Description |
|---|---|---|---|---|
| `model` | str |  | ✓ | The model that will complete your prompt.\n\nSee [models](https://docs.anthropic.com/en/docs/models-overview) for additional details and options. |
| `messages` | list |  | ✓ | Input messages. Our models are trained to operate on alternating `user` and `assistant` conversational turns. When creating a new `Message`, you specify the prior conversational turns with the `messages` parameter, and the model then generates the next `Message` in the conversation. Consecutive `user` or `assistant` turns in your request will be combined into a single turn. Each input message must be an object with a `role` and `content`. You can specify a single `user`-role message, or you can include multiple `user` and `assistant` messages. If the final message uses the `assistant` role, the response content will continue immediately from the content in that message. This can be used to constrain part of the model's response. Example with a single `user` message: ```json [{"role": "user", "content": "Hello, Claude"}] ``` Example with multiple conversational turns: ```json [ {"role": "user", "content": "Hello there."}, {"role": "assistant", "content": "Hi, I'm Claude. How can I help you?"}, {"role": "user", "content": "Can you explain LLMs in plain English?"}, ] ``` Example with a partially-filled response from Claude: ```json [ {"role": "user", "content": "What's the Greek name for Sun? (A) Sol (B) Helios (C) Sun"}, {"role": "assistant", "content": "The best answer is ("}, ] ``` Each input message `content` may be either a single `string` or an array of content blocks, where each block has a specific `type`. Using a `string` for `content` is shorthand for an array of one content block of type `"text"`. The following input messages are equivalent: ```json {"role": "user", "content": "Hello, Claude"} ``` ```json {"role": "user", "content": [{"type": "text", "text": "Hello, Claude"}]} ``` See [input examples](https://docs.claude.com/en/api/messages-examples). Note that if you want to include a [system prompt](https://docs.claude.com/en/docs/system-prompts), you can use the top-level `system` parameter — there is no `"system"` role for input messages in the Messages API. There is a limit of 100,000 messages in a single request. |
| `cache_control` | dict | None |  | Top-level cache control automatically applies a cache_control marker to the last cacheable block in the request. |
| `container` | str | None |  | Container identifier for reuse across requests. |
| `inference_geo` | str | None |  | Specifies the geographic region for inference processing. If not specified, the workspace's `default_inference_geo` is used. |
| `max_tokens` | int |  | ✓ | The maximum number of tokens to generate before stopping. Note that our models may stop _before_ reaching this maximum. This parameter only specifies the absolute maximum number of tokens to generate. Different models have different maximum values for this parameter. See [models](https://docs.claude.com/en/docs/models-overview) for details. |
| `metadata` | dict |  |  | An object describing metadata about the request. |
| `output_config` | dict |  |  | Configuration options for the model's output, such as the output format. |
| `service_tier` | str |  |  | Determines whether to use priority capacity (if available) or standard capacity for this request. Anthropic offers different levels of service for your API requests. See [service-tiers](https://docs.claude.com/en/api/service-tiers) for details. |
| `stop_sequences` | list |  |  | Custom text sequences that will cause the model to stop generating. Our models will normally stop when they have naturally completed their turn, which will result in a response `stop_reason` of `"end_turn"`. If you want the model to stop generating when it encounters custom strings of text, you can use the `stop_sequences` parameter. If the model encounters one of the custom sequences, the response `stop_reason` value will be `"stop_sequence"` and the response `stop_sequence` value will contain the matched stop sequence. |
| `stream` | bool |  |  | Whether to incrementally stream the response using server-sent events. See [streaming](https://docs.claude.com/en/api/messages-streaming) for details. |
| `system` | None |  |  | System prompt. A system prompt is a way of providing context and instructions to Claude, such as specifying a particular goal or role. See our [guide to system prompts](https://docs.claude.com/en/docs/system-prompts). |
| `temperature` | float |  |  | Amount of randomness injected into the response. Defaults to `1.0`. Ranges from `0.0` to `1.0`. Use `temperature` closer to `0.0` for analytical / multiple choice, and closer to `1.0` for creative and generative tasks. Note that even with `temperature` of `0.0`, the results will not be fully deterministic. |
| `thinking` | dict |  |  | Configuration for enabling Claude's extended thinking. When enabled, responses include `thinking` content blocks showing Claude's thinking process before the final answer. Requires a minimum budget of 1,024 tokens and counts towards your `max_tokens` limit. See [extended thinking](https://docs.claude.com/en/docs/build-with-claude/extended-thinking) for details. |
| `tool_choice` | dict |  |  | How the model should use the provided tools. The model can use a specific tool, any available tool, decide by itself, or not use tools at all. |
| `tools` | list |  |  | Definitions of tools that the model may use. If you include `tools` in your API request, the model may return `tool_use` content blocks that represent the model's use of those tools. You can then run those tools using the tool input generated by the model and then optionally return results back to the model using `tool_result` content blocks. There are two types of tools: **client tools** and **server tools**. The behavior described below applies to client tools. For [server tools](https://docs.claude.com/en/docs/agents-and-tools/tool-use/overview\#server-tools), see their individual documentation as each has its own behavior (e.g., the [web search tool](https://docs.claude.com/en/docs/agents-and-tools/tool-use/web-search-tool)). Each tool definition includes: * `name`: Name of the tool. * `description`: Optional, but strongly-recommended description of the tool. * `input_schema`: [JSON schema](https://json-schema.org/draft/2020-12) for the tool `input` shape that the model will produce in `tool_use` output content blocks. For example, if you defined `tools` as: ```json [ { "name": "get_stock_price", "description": "Get the current stock price for a given ticker symbol.", "input_schema": { "type": "object", "properties": { "ticker": { "type": "string", "description": "The stock ticker symbol, e.g. AAPL for Apple Inc." } }, "required": ["ticker"] } } ] ``` And then asked the model "What's the S&P 500 at today?", the model might produce `tool_use` content blocks in the response like this: ```json [ { "type": "tool_use", "id": "toolu_01D7FLrfh4GYq7yT1ULFeyMV", "name": "get_stock_price", "input": { "ticker": "^GSPC" } } ] ``` You might then run your `get_stock_price` tool with `{"ticker": "^GSPC"}` as an input, and return the following back to the model in a subsequent `user` message: ```json [ { "type": "tool_result", "tool_use_id": "toolu_01D7FLrfh4GYq7yT1ULFeyMV", "content": "259.75 USD" } ] ``` Tools can be used for workflows that include running client-side tools and functions, or more generally whenever you want the model to produce a particular JSON structure of output. See our [guide](https://docs.claude.com/en/docs/tool-use) for more details. |
| `top_k` | int |  |  | Only sample from the top K options for each subsequent token. Used to remove "long tail" low probability responses. [Learn more technical details here](https://towardsdatascience.com/how-to-sample-from-language-models-682bceb97277). Recommended for advanced use cases only. You usually only need to use `temperature`. |
| `top_p` | float |  |  | Use nucleus sampling. In nucleus sampling, we compute the cumulative distribution over all the options for each subsequent token in decreasing probability order and cut it off once it reaches a particular probability specified by `top_p`. You should either alter `temperature` or `top_p`, but not both. Recommended for advanced use cases only. You usually only need to use `temperature`. |


## Discovery

In [ ]:
gem_spec = dict2obj((specs_path/'gemini.json').read_json())

In [ ]:
gem_spec.keys()

dict_keys(['title', 'ownerDomain', 'batchPath', 'kind', 'version_module', 'baseUrl', 'fullyEncodeReservedExpansion', 'schemas', 'protocol', 'version', 'name', 'id', 'basePath', 'mtlsRootUrl', 'parameters', 'icons', 'resources', 'description', 'rootUrl', 'documentationLink', 'ownerName', 'servicePath', 'discoveryVersion', 'canonicalName', 'revision', 'auth'])

In [ ]:
gem_spec.name, gem_spec.version, gem_spec.title, gem_spec.description, gem_spec.baseUrl

('generativelanguage',
 'v1beta',
 'Generative Language API',
 'The Gemini API allows developers to build generative AI applications using Gemini models. Gemini is our most capable model, built from the ground up to be multimodal. It can generalize and seamlessly understand, operate across, and combine different types of information including language, images, audio, video, and code. You can use the Gemini API for use cases like reasoning across text and images, content generation, dialogue agents, summarization and classification systems, and more.',
 'https://generativelanguage.googleapis.com/')

In [ ]:
gem_spec.icons, gem_spec.documentationLink

({'x16': 'http://www.google.com/images/icons/product/search-16.gif',
  'x32': 'http://www.google.com/images/icons/product/search-32.gif'},
 'https://developers.generativeai.google/api')

In [ ]:
len(gem_spec.schemas), len(gem_spec.resources)

(177, 10)

In [ ]:
list(gem_spec.schemas)[:10]

['ListOperationsResponse',
 'Operation',
 'Status',
 'Empty',
 'GenerateContentRequest',
 'Content',
 'Part',
 'Blob',
 'FunctionCall',
 'FunctionResponse']

In [ ]:
list(gem_spec.resources)

['batches',
 'models',
 'tunedModels',
 'dynamic',
 'cachedContents',
 'media',
 'files',
 'generatedFiles',
 'fileSearchStores',
 'corpora']

In [ ]:
tmpl = "https://{API}/$discovery/rest?version={VERSION}"

In [ ]:
list(gem_spec.resources.models.methods)

['generateContent',
 'generateAnswer',
 'streamGenerateContent',
 'embedContent',
 'batchEmbedContents',
 'countTokens',
 'batchGenerateContent',
 'asyncBatchEmbedContent',
 'generateMessage',
 'countMessageTokens',
 'get',
 'list',
 'predict',
 'predictLongRunning',
 'generateText',
 'embedText',
 'batchEmbedText',
 'countTextTokens']

In [ ]:
gem_spec.resources.models.methods.generateContent

```python
{ 'description': 'Generates a model response given an input '
                 '`GenerateContentRequest`. Refer to the [text generation '
                 'guide](https://ai.google.dev/gemini-api/docs/text-generation) '
                 'for detailed usage information. Input capabilities differ '
                 'between models, including tuned models. Refer to the [model '
                 'guide](https://ai.google.dev/gemini-api/docs/models/gemini) '
                 'and [tuning '
                 'guide](https://ai.google.dev/gemini-api/docs/model-tuning) '
                 'for details.',
  'flatPath': 'v1beta/models/{modelsId}:generateContent',
  'httpMethod': 'POST',
  'id': 'generativelanguage.models.generateContent',
  'parameterOrder': ['model'],
  'parameters': { 'model': { 'description': 'Required. The name of the `Model` '
                                            'to use for generating the '
                                            'completion. Format: '
                                            '`models/{model}`.',
                             'location': 'path',
                             'pattern': '^models/[^/]+$',
                             'required': True,
                             'type': 'string'}},
  'path': 'v1beta/{+model}:generateContent',
  'request': {'$ref': 'GenerateContentRequest'},
  'response': {'$ref': 'GenerateContentResponse'}}
```

In [ ]:
list(gem_spec.schemas.GenerateContentRequest.properties)

['model',
 'systemInstruction',
 'contents',
 'tools',
 'toolConfig',
 'safetySettings',
 'generationConfig',
 'cachedContent',
 'serviceTier',
 'store']

In [ ]:
gem_spec.schemas.GenerateContentRequest.properties

```python
{ 'cachedContent': { 'description': 'Optional. The name of the content '
                                    '[cached](https://ai.google.dev/gemini-api/docs/caching) '
                                    'to use as context to serve the '
                                    'prediction. Format: '
                                    '`cachedContents/{cachedContent}`',
                     'type': 'string'},
  'contents': { 'description': 'Required. The content of the current '
                               'conversation with the model. For single-turn '
                               'queries, this is a single instance. For '
                               'multi-turn queries like '
                               '[chat](https://ai.google.dev/gemini-api/docs/text-generation#chat), '
                               'this is a repeated field that contains the '
                               'conversation history and the latest request.',
                'items': {'$ref': 'Content'},
                'type': 'array'},
  'generationConfig': { '$ref': 'GenerationConfig',
                        'description': 'Optional. Configuration options for '
                                       'model generation and outputs.'},
  'model': { 'description': 'Required. The name of the `Model` to use for '
                            'generating the completion. Format: '
                            '`models/{model}`.',
             'type': 'string'},
  'safetySettings': { 'description': 'Optional. A list of unique '
                                     '`SafetySetting` instances for blocking '
                                     'unsafe content. This will be enforced on '
                                     'the `GenerateContentRequest.contents` '
                                     'and '
                                     '`GenerateContentResponse.candidates`. '
                                     'There should not be more than one '
                                     'setting for each `SafetyCategory` type. '
                                     'The API will block any contents and '
                                     'responses that fail to meet the '
                                     'thresholds set by these settings. This '
                                     'list overrides the default settings for '
                                     'each `SafetyCategory` specified in the '
                                     'safety_settings. If there is no '
                                     '`SafetySetting` for a given '
                                     '`SafetyCategory` provided in the list, '
                                     'the API will use the default safety '
                                     'setting for that category. Harm '
                                     'categories HARM_CATEGORY_HATE_SPEECH, '
                                     'HARM_CATEGORY_SEXUALLY_EXPLICIT, '
                                     'HARM_CATEGORY_DANGEROUS_CONTENT, '
                                     'HARM_CATEGORY_HARASSMENT, '
                                     'HARM_CATEGORY_CIVIC_INTEGRITY are '
                                     'supported. Refer to the '
                                     '[guide](https://ai.google.dev/gemini-api/docs/safety-settings) '
                                     'for detailed information on available '
                                     'safety settings. Also refer to the '
                                     '[Safety '
                                     'guidance](https://ai.google.dev/gemini-api/docs/safety-guidance) '
                                     'to learn how to incorporate safety '
                                     'considerations in your AI applications.',
                      'items': {'$ref': 'SafetySetting'},
                      'type': 'array'},
  'serviceTier': { 'description': 'Optional. The service tier of the request.',
                   'enum': ['SERVICE_TIER_UNSPECIFIED', 'SERVICE_TIER_STANDARD', 'SERVICE_TIER_FLEX', 'SERVICE_TIER_PRIORITY'],
                   'enumDescriptions': ['Default service tier, which is standard.', 'Standard service tier.', 'Flex service tier.', 'Priority service tier.'],
                   'type': 'string'},
  'store': { 'description': 'Optional. Configures the logging behavior for a '
                            'given request. If set, it takes precedence over '
                            'the project-level logging config.',
             'type': 'boolean'},
  'systemInstruction': { '$ref': 'Content',
                         'description': 'Optional. Developer set [system '
                                        'instruction(s)](https://ai.google.dev/gemini-api/docs/system-instructions). '
                                        'Currently, text only.'},
  'toolConfig': { '$ref': 'ToolConfig',
                  'description': 'Optional. Tool configuration for any `Tool` '
                                 'specified in the request. Refer to the '
                                 '[Function calling '
                                 'guide](https://ai.google.dev/gemini-api/docs/function-calling#function_calling_mode) '
                                 'for a usage example.'},
  'tools': { 'description': 'Optional. A list of `Tools` the `Model` may use '
                            'to generate the next response. A `Tool` is a '
                            'piece of code that enables the system to interact '
                            'with external systems to perform an action, or '
                            'set of actions, outside of knowledge and scope of '
                            'the `Model`. Supported `Tool`s are `Function` and '
                            '`code_execution`. Refer to the [Function '
                            'calling](https://ai.google.dev/gemini-api/docs/function-calling) '
                            'and the [Code '
                            'execution](https://ai.google.dev/gemini-api/docs/code-execution) '
                            'guides to learn more.',
             'items': {'$ref': 'Tool'},
             'type': 'array'}}
```

### `disovery_to_ops`


In [ ]:
#| export
def _discovery_collect_params(m, spec):
    "Collect route and query params from a Discovery method dict."
    route, query, req, ptypes, pdocs, defaults = [], [], set(), {}, {}, {}
    for nm, pd in (m.get("parameters") or {}).items():
        if not isinstance(pd, dict): continue
        loc = str(pd.get("location") or "")
        if loc == "path" and nm not in route: route.append(nm); req.add(nm)
        elif loc == "query" and nm not in query:
            query.append(nm)
            if pd.get("required"): req.add(nm)
        ptypes[nm] = _schema_py_type(pd, spec)
        desc = _clean_desc(pd.get("description"))
        if desc: pdocs[nm] = desc
    return AttrDict(route_params=route, query_params=query, required_params=req,
                param_types=ptypes, param_docs=pdocs, param_defaults=defaults)

def _discovery_body_params(m, schemas, spec):
    "Extract body params from a Discovery method's request field."
    req_obj = m.get("request", {})
    ref = req_obj.get("$ref")
    schema = schemas.get(ref, {}) if ref else req_obj
    props, req = _schema_props_required(schema, spec)
    ptypes = {k: _schema_py_type(v, spec) for k,v in props.items()}
    pdocs = {k: d for k,v in props.items() if (d := _prop_desc(v, spec))}
    defaults = {k: d for k,v in props.items() if (d := _prop_default(v, spec)) is not _MISSING}
    return AttrDict(body_params=list(props), required_params=req, param_types=ptypes,
                param_docs=pdocs, param_defaults=defaults)

In [ ]:
#| export
def discovery_to_ops(spec):
    "Convert Google Discovery spec to OpSpec list."
    schemas = spec.get("schemas", {})
    ops = []
    def walk(res, group):
        for mname, m in res.get("methods", {}).items():
            verb = m.get("httpMethod", "").upper()
            if verb.lower() not in _http_verbs: continue
            path = m.get("path")
            pdict = _discovery_collect_params(m, spec)
            bpdict = _discovery_body_params(m, schemas, spec)
            ops.append(OpSpec(
                group=group, name=snake(mname), path=path, verb=verb,
                summary=_clean_desc(m.get("description", "")),
                route_params=pdict.route_params or _path_params(path),
                query_params=pdict.query_params,
                body_params=bpdict.body_params,
                required_params=pdict.required_params | bpdict.required_params,
                param_types=merge(pdict.param_types, bpdict.param_types),
                param_defaults=merge(pdict.param_defaults, bpdict.param_defaults),
                param_docs=merge(pdict.param_docs, bpdict.param_docs),
                docs_url=str(m.get("documentationLink", ""))
            ))
        for rname, child in res.get("resources", {}).items(): walk(child, group + [rname])
    for rname, res in spec.get("resources", {}).items(): walk(res, [rname])
    return ops

In [ ]:
gem_spec.keys()

dict_keys(['title', 'ownerDomain', 'batchPath', 'kind', 'version_module', 'baseUrl', 'fullyEncodeReservedExpansion', 'schemas', 'protocol', 'version', 'name', 'id', 'basePath', 'mtlsRootUrl', 'parameters', 'icons', 'resources', 'description', 'rootUrl', 'documentationLink', 'ownerName', 'servicePath', 'discoveryVersion', 'canonicalName', 'revision', 'auth'])

In [ ]:
gem_ops = discovery_to_ops(gem_spec)

In [ ]:
len(gem_ops)

79

In [ ]:
gem_ops[10]

| Field | Value |
|---|---|
| `group` | ['models'] |
| `name` | batch_embed_contents |
| `path` | v1beta/{+model}:batchEmbedContents |
| `verb` | POST |
| `summary` | Generates multiple embedding vectors from the input `Content` which consists of a batch of strings represented as `EmbedContentRequest` objects. |
| `route_params` | ['model'] |
| `query_params` | [] |
| `body_params` | ['requests'] |
| `file_params` | [] |
| `required_params` | {'model', 'requests'} |
| `docs_url` |  |

| Param | Type | Default | Required | Description |
|---|---|---|---|---|
| `model` | str |  | ✓ | Required. The model's resource name. This serves as an ID for the Model to use. This name should match a model name returned by the `ListModels` method. Format: `models/{model}` |
| `requests` | list |  | ✓ | Required. Embed requests for the batch. The model in each of these requests must match the model specified `BatchEmbedContentsRequest.model`. |


### SpecParser

A thin class that sets `base_url` and `ops` for openapi and discovery specs using 2 methods `from_openapi` and `from_discovery`

In [ ]:
#| export
class SpecParser:
    "Parse OpenAPI or Discovery specs into a unified `base_url` + `ops` representation."
    def __init__(self, base_url, ops): store_attr()

    @classmethod
    def from_openapi(cls, spec):
        "Create from an OpenAPI spec dict."
        base_url = first((spec.get("servers") or [{}]), {}).get("url", "")
        return cls(base_url=base_url, ops=openapi_to_ops(spec))

    @classmethod
    def from_discovery(cls, spec):
        "Create from a Google Discovery spec dict."
        return cls(base_url=spec.get("baseUrl", ""), ops=discovery_to_ops(spec))

    # TODO: Ideally this should show how to init the object 
    def __repr__(self):
        return f"SpecParser(base_url={self.base_url!r}, ops={len(self.ops)})"

In [ ]:
gh_ops     = SpecParser.from_openapi(gh_spec)
stripe_ops = SpecParser.from_openapi(stripe_spec)
oai_ops = SpecParser.from_openapi(oai_spec)
ant_ops = SpecParser.from_openapi(ant_spec)
gem_ops = SpecParser.from_discovery(gem_spec)

In [ ]:
gh_ops, stripe_ops, oai_ops, ant_ops, gem_ops

(SpecParser(base_url='https://api.github.com', ops=1112),
 SpecParser(base_url='https://api.stripe.com/', ops=587),
 SpecParser(base_url='https://api.openai.com/v1', ops=241),
 SpecParser(base_url='https://api.anthropic.com', ops=47),
 SpecParser(base_url='https://generativelanguage.googleapis.com/', ops=79))

## Export -

In [ ]:
#|hide
#|eval: false
import nbdev; nbdev.nbdev_export()